# Mirage — DistilBERT fine-tuning (Kaggle / Colab, T4 GPU)

Thin notebook: all logic lives in the `mirage` package. Steps:
1. Turn on **GPU T4** and **Internet** in the notebook settings.
2. `REPO_URL` below points at the project repo (change it if you fork).
3. Run all cells. Training takes about 10–25 minutes per seed; predictions about 5 minutes per seed.
4. Download `distilbert_outputs.zip` from the output panel and unzip it into the **repo root** on your laptop
   (it contains `artifacts/models/distilbert__*` and `results/predictions/distilbert__*`).

In [ ]:
REPO_URL = "https://github.com/AditeyaAItronics/mirage.git"
SEEDS = [13, 42, 87]  # fallback if short on GPU time: [42]
SMOKE_TEST = False  # True = 2,000 examples, ~2 minutes, to check everything works first

In [ ]:
import os, subprocess
os.chdir("/kaggle/working") if os.path.isdir("/kaggle/working") else None
if not os.path.isdir("mirage"):
    subprocess.run(["git", "clone", REPO_URL, "mirage"], check=True)
os.chdir("mirage")
!pip install -q -e ".[dl]"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU!")

In [ ]:
!python -m mirage prepare

In [ ]:
limit = "--limit 2000" if SMOKE_TEST else ""
seeds = " ".join(map(str, SEEDS))
!python -m mirage train-distilbert --seeds {seeds} {limit}

In [ ]:
# Predict on the GPU: 10,000 IMDb reviews at 512 tokens x 3 seeds is slow on a laptop CPU.
!python -m mirage build-suite
!python -m mirage predict --models distilbert


In [ ]:
# Package final weights + DistilBERT predictions (no checkpoints). Unzip into the repo root.
import glob, shutil, zipfile
for d in glob.glob("artifacts/models/distilbert__*/checkpoints"):
    shutil.rmtree(d)
out = "/kaggle/working/distilbert_outputs.zip" if os.path.isdir("/kaggle/working") else "distilbert_outputs.zip"
files = glob.glob("artifacts/models/distilbert__*/**", recursive=True) + glob.glob("results/predictions/distilbert__*.parquet")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.isfile(f):
            z.write(f)
print(out, os.path.getsize(out) // 2**20, "MB")
!ls -la artifacts/models results/predictions | head -40